In [1]:
# bowaka_v2_lab notebook bootstrap cell — DO NOT EDIT BY HAND.
# Adds the lab's src/ (and its bowaka_common dependency) to sys.path and pins
# the working directory to the repo root, so `import bowaka_v2_lab` and
# repo-root-relative CONFIG_PATH parameters resolve identically under jupyter,
# papermill, and the QuantsLab scheduler.
import os
import sys
from pathlib import Path

_lab_root = None
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "bowaka_v2_lab" / "__init__.py").is_file():
        _lab_root = _candidate
        break
if _lab_root is None:
    raise RuntimeError(
        f"bowaka_v2_lab bootstrap: src/bowaka_v2_lab/ not found at or above {Path.cwd()}"
    )

# Pin CWD to the repo root (the directory holding research_notebooks/ and the
# Makefile) so repo-root-relative CONFIG_PATH values resolve regardless of how
# the notebook was launched (jupyter CWD = notebook dir, scheduler = repo root).
_repo_root = _lab_root
for _candidate in [_lab_root, *_lab_root.parents]:
    if (_candidate / "research_notebooks").is_dir() and (_candidate / "Makefile").is_file():
        _repo_root = _candidate
        break
os.chdir(_repo_root)

# Make the lab and its bowaka_common dependency importable from the working
# tree, even when the packages are not pip-installed. v1 bowaka_lab is
# deliberately excluded — v2 must not import v1.
for _src in (_lab_root / "src",
             _repo_root / "research_notebooks" / "bowaka_common" / "src"):
    if _src.is_dir() and str(_src) not in sys.path:
        sys.path.insert(0, str(_src))

import bowaka_v2_lab  # noqa: F401
print(f"bowaka_v2_lab {bowaka_v2_lab.__version__} (cwd={_repo_root})")


bowaka_v2_lab 0.1.0 (cwd=/quants-lab)


In [2]:
# Papermill parameter cell.
CONFIG_PATH = 'research_notebooks/bowaka_v2_lab/configs/bowaka_v2_backtest_smoke.yml'


# 06 — Execution Cost & Liquidity Study

Runs a real backtest, then profiles its entry decisions by ADV bucket, spread bucket, and time of day.

In [3]:
import pandas as pd
from bowaka_v2_lab.backtest_runner import run_config_backtest
from bowaka_v2_lab.reports.liquidity_execution import (adv_bucket_distribution,
  spread_bucket_distribution, time_of_day_buckets)
result = run_config_backtest(CONFIG_PATH)
print('backtest summary:', result.summary)
dec_path = result.run_dir / 'entry_decisions.parquet'
if dec_path.is_file():
    decisions = pd.read_parquet(dec_path)
    print('entry decisions:', len(decisions))
    print('--- ADV-bucket distribution ---')
    print(adv_bucket_distribution(decisions))
    print('--- spread-bucket distribution ---')
    print(spread_bucket_distribution(decisions))
    print('--- time-of-day distribution ---')
    print(time_of_day_buckets(decisions))
else:
    print('no entry_decisions.parquet at', dec_path)


backtest summary: {'schema_version': 1, 'run_id': '20260523_bowaka_v2_backtest_e809609e_c7bfe6ed', 'strategy_id': 'bowaka_v2', 'strategy_version': '0.1.0', 'feed': 'iex', 'cost_stress': 'base', 'initial_bankroll': 100000.0, 'final_bankroll': 99889.45735600003, 'net_return_pct': -0.0011054264399997192, 'total_pnl': -110.5426440000004, 'max_drawdown_pct': 0.0011054264400000102, 'n_trades': 6, 'win_rate': 0.0, 'avg_win': 0.0, 'avg_loss': -18.423774000000066, 'candidate_events_count': 441, 'entry_decisions_count': 441, 'accepted_count': 6, 'rejected_count': 435, 'broker_reject_count': 0, 'ambiguous_bar_count': 6, 'missing_quote_count': 0, 'fills_count': 6, 'partial_fill_count': 0, 'fill_rate': 1.0, 'historical_quote_coverage_pct': 0.0, 'fees_paid_total': 0.0, 'protection': {'max_unprotected_seconds_observed': 0.0, 'oco_attach_attempts_count': 0, 'oco_attach_failure_count': 0, 'fallback_stop_count': 0, 'flatten_unprotected_count': 0, 'entries_blocked_by_protection_count': 0, 'total_unprotec